In [0]:
df = spark.table("bronze_complaints")
#df.printSchema()          # column names + types (all strings right now)
#print(df.count())         # 84,619

from pyspark.sql import functions as F

# columns whose text values we clean the same way
text_cols = ["product", "sub_product", "issue", "sub_issue", "company",
             "state", "submitted_via", "company_response_to_consumer",
             "company_public_response", "tags"]

silver = df
for c in text_cols:
    silver = silver.withColumn(
        c,
        F.when(F.trim(F.col(c)).isin("", "None", "N/A"), None)   # blanks/"None" -> real NULL
         .otherwise(F.trim(F.col(c)))                              # strip extra spaces
    )

silver = (silver
    # data types
    .withColumn("complaint_id",          F.col("complaint_id").cast("bigint"))
    .withColumn("date_received",         F.to_date(F.col("date_received")))
    .withColumn("date_sent_to_company",  F.to_date(F.col("date_sent_to_company")))
    .withColumn("timely_response_flag",  F.when(F.col("timely_response") == "Yes", True)
                                          .when(F.col("timely_response") == "No", False))
    # standardize
    .withColumn("company", F.upper(F.col("company")))
    .withColumn("state",   F.upper(F.col("state")))
    # derived columns
    .withColumn("days_to_company", F.datediff("date_sent_to_company", "date_received"))
    #.withColumn("has_narrative",   F.col("consumer_complaint_narrative").isNotNull())
    # audit column
    .withColumn("load_ts", F.current_timestamp())
)

from pyspark.sql import Window

w = Window.partitionBy("complaint_id").orderBy(
        F.col("date_received").desc(),          # latest date first
        F.col("date_sent_to_company").desc()    # tie-breaker
    )

before = silver.count()
silver = (silver
          .withColumn("rn", F.row_number().over(w))   # 1 = latest row per complaint
          .filter(F.col("rn") == 1)
          .drop("rn"))
after = silver.count()
print(f"Rows before dedupe: {before}, after: {after}, duplicates removed: {before - after}")

silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_complaints")
display(spark.table("silver_complaints").limit(10))
